In [2]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import RobertaTokenizer, RobertaForSequenceClassification
from torch.optim import AdamW
import numpy as np
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt

# Set device (use GPU if available, otherwise CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(torch.cuda.is_available())  # Must return True
print(torch.cuda.get_device_name(0))  # Your GPU model

Using device: cuda
True
NVIDIA GeForce RTX 2050


In [9]:
train_path = "emotion_intensity_data/train.csv"
dev_path   = "emotion_intensity_data/dev.csv"
test_path  = "emotion_intensity_data/test.csv"

def load_dataset(file_path):
    df = pd.read_csv(file_path)
    print(f"Loaded {file_path} with {len(df)} samples")
    return df

train_df = load_dataset(train_path)
dev_df   = load_dataset(dev_path)
test_df  = load_dataset(test_path)

print("\nColumns:", train_df.columns.tolist())
print("\nSample data:\n", train_df.head())


Loaded emotion_intensity_data/train.csv with 2768 samples
Loaded emotion_intensity_data/dev.csv with 116 samples
Loaded emotion_intensity_data/test.csv with 2767 samples

Columns: ['id', 'text', 'anger', 'fear', 'joy', 'sadness', 'surprise']

Sample data:
                         id                                               text  \
0  eng_train_track_b_00001                       Colorado, middle of nowhere.   
1  eng_train_track_b_00002  This involved swimming a pretty large lake tha...   
2  eng_train_track_b_00003        It was one of my most shameful experiences.   
3  eng_train_track_b_00004  After all, I had vegetables coming out my ears...   
4  eng_train_track_b_00005                        Then the screaming started.   

   anger  fear  joy  sadness  surprise  
0      0     1    0        0         1  
1      0     2    0        0         0  
2      0     1    0        3         0  
3      0     0    0        0         0  
4      0     3    0        1         2  


In [10]:
ALL_EMOTIONS = ["anger", "surprise", "disgust", "fear", "joy", "sadness"]

# Use only those emotions that exist as columns in your CSV
EMOTIONS = [e for e in ALL_EMOTIONS if e in train_df.columns]

print("Emotion columns found:", EMOTIONS)

Emotion columns found: ['anger', 'surprise', 'fear', 'joy', 'sadness']


In [11]:
ALL_EMOTIONS = ["anger", "fear", "joy", "sadness", "surprise"]
INTENSITIES = [0, 1, 2, 3]

# 20 labels
LABELS = [f"{e}_{i}" for e in ALL_EMOTIONS for i in INTENSITIES]
label2id = {label: idx for idx, label in enumerate(LABELS)}
id2label = {idx: label for label, idx in label2id.items()}

print("Total classes:", len(LABELS))   # should print 20
print("First few labels:", LABELS[:8])

Total classes: 20
First few labels: ['anger_0', 'anger_1', 'anger_2', 'anger_3', 'fear_0', 'fear_1', 'fear_2', 'fear_3']


In [12]:
TEXT_COL = "text"

def row_to_label_text(row):
    scores = {e: int(row[e]) for e in ALL_EMOTIONS}
    max_val = max(scores.values())

    # candidate emotions (in case of tie)
    candidates = [e for e, v in scores.items() if v == max_val]

    # tie-break by fixed order in ALL_EMOTIONS
    for e in ALL_EMOTIONS:
        if e in candidates:
            chosen_emotion = e
            break

    return f"{chosen_emotion}_{max_val}"

def prepare_df(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df = df.dropna(subset=[TEXT_COL])

    # ensure emotion columns are ints
    for e in ALL_EMOTIONS:
        df[e] = df[e].astype(int)

    # create single-step label
    df["label_text"] = df.apply(row_to_label_text, axis=1)
    df["label"] = df["label_text"].map(label2id).astype(int)

    return df

train_df = prepare_df(train_df)
dev_df   = prepare_df(dev_df)
test_df  = prepare_df(test_df)

print(train_df[[TEXT_COL, "label_text", "label"]].head())

                                                text label_text  label
0                       Colorado, middle of nowhere.     fear_1      5
1  This involved swimming a pretty large lake tha...     fear_2      6
2        It was one of my most shameful experiences.  sadness_3     15
3  After all, I had vegetables coming out my ears...    anger_0      0
4                        Then the screaming started.     fear_3      7


In [35]:
class EmotionIntensityDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=128):
        self.texts = dataframe['text'].tolist()
        # Get emotion intensity columns (adjust these names based on your CSV)
        self.anger = dataframe['anger'].tolist()
        self.fear = dataframe['fear'].tolist()
        self.joy = dataframe['joy'].tolist()
        self.sadness = dataframe['sadness'].tolist()
        self.surprise = dataframe['surprise'].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        
        # Get emotion intensities for this text (0-3 scale)
        emotions = torch.tensor([
            self.anger[idx],
            self.fear[idx],
            self.joy[idx],
            self.sadness[idx],
            self.surprise[idx]
        ], dtype=torch.long)
        
        # Tokenize the text (convert to numbers)
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': emotions
        }

In [38]:
# Step 2: Initialize tokenizer
tokenizer = RobertaTokenizer.from_pretrained('roberta-base')

# Step 3: Create datasets
if train_df is not None and dev_df is not None:
    train_dataset = EmotionIntensityDataset(train_df, tokenizer)
    dev_dataset = EmotionIntensityDataset(dev_df, tokenizer)
    test_dataset = EmotionIntensityDataset(test_df, tokenizer)

    # Step 4: Create data loaders
    batch_size = 8
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    dev_loader = DataLoader(dev_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    print(f"\nTraining batches: {len(train_loader)}")
    print(f"Validation batches: {len(dev_loader)}")
    print(f"Test batches: {len(test_loader)}")

    class EmotionIntensityModel(nn.Module):
        def __init__(self):
            super(EmotionIntensityModel, self).__init__()
            
            # Use pre-trained RoBERTa model
            self.roberta = RobertaForSequenceClassification.from_pretrained('roberta-base',num_labels=5 ) # We predict 5 emotions
        
        def forward(self, input_ids, attention_mask):
            # Pass text through the model
            outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
            return outputs.logits
        
    model = EmotionIntensityModel()
    model = model.to(device)
    print(f"\nModel created and moved to {device}")

    # Step 6: Set up training components
    optimizer = AdamW(model.parameters(), lr=2e-5)  # Learning rate
    loss_fn = nn.CrossEntropyLoss()  # Loss function
   
    def train_model(model, train_loader, dev_loader, epochs=3):
        train_losses = []
        dev_losses = []
        
        for epoch in range(epochs):
            print(f"\nEpoch {epoch + 1}/{epochs}")
            print("-" * 30)
            
            # Training phase
            model.train()
            total_train_loss = 0
            
            for batch_idx, batch in enumerate(train_loader):
                # Move data to device
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['labels'].to(device)
                
                # Reset gradients
                optimizer.zero_grad()
                
                # Forward pass (make predictions)
                predictions = model(input_ids, attention_mask)
                
                # Calculate loss (how wrong are predictions)
                loss = 0
                for i in range(5):  # For each emotion
                    loss += loss_fn(predictions[:, i, :], labels[:, i])
                
                # Backward pass (learn from mistakes)
                loss.backward()
                
                # Update model weights
                optimizer.step()
                
                total_train_loss += loss.item()
                
                # Print progress every 50 batches
                if batch_idx % 50 == 0:
                    print(f"  Batch {batch_idx}/{len(train_loader)}, Loss: {loss.item():.4f}")


            avg_train_loss = total_train_loss / len(train_loader)
            train_losses.append(avg_train_loss)
            
            # Validation phase
            model.eval()
            total_dev_loss = 0
            all_predictions = []
            all_labels = []
            
            with torch.no_grad():  # Don't calculate gradients for validation
                for batch in dev_loader:
                    input_ids = batch['input_ids'].to(device)
                    attention_mask = batch['attention_mask'].to(device)
                    labels = batch['labels'].to(device)
                    
                    predictions = model(input_ids, attention_mask)
                    
                    # Calculate loss
                    batch_loss = 0
                    for i in range(5):
                        batch_loss += loss_fn(predictions[:, i, :], labels[:, i])
                    
                    total_dev_loss += batch_loss.item()
                    
                    # Get predicted classes
                    pred_classes = torch.argmax(predictions, dim=2)
                    all_predictions.extend(pred_classes.cpu().numpy())
                    all_labels.extend(labels.cpu().numpy())
            
            avg_dev_loss = total_dev_loss / len(dev_loader)
            dev_losses.append(avg_dev_loss)
            
            # Calculate accuracy
            accuracy = np.mean(np.array(all_predictions) == np.array(all_labels))
            
            print(f"Train Loss: {avg_train_loss:.4f}")
            print(f"Dev Loss: {avg_dev_loss:.4f}")
            print(f"Dev Accuracy: {accuracy:.4f}")
        
        return train_losses, dev_losses
        

# Train the model
train_losses, dev_losses = train_model(model, train_loader, dev_loader, epochs=3)




Training batches: 346
Validation batches: 15
Test batches: 346


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Model created and moved to cuda

Epoch 1/3
------------------------------


c:\Users\Sharayu\Python313\Lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `RobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


IndexError: too many indices for tensor of dimension 2